# Data Modeling

Akan dilakukan pemodelan klasifikasi teks menggunakan SVM, RNN, dan LSTM dengan tiga teknik feature extraction, yaitu BoW, TF-IDF, dan Word2Vec, untuk membandingkan kinerja dan menentukan model terbaik.

Install library yang diperlukan:

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.5 MB/s eta 0:00:00


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 78.4 MB/s eta 0:00:00


Import library yang dibutuhkan:

In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from gensim.models import Word2Vec
import warnings
warnings.filterwarnings('ignore')

# Load Data

In [ ]:
df = pd.read_csv('/content/dana_preprocessing.csv')
df_selected = df[['reviewId', 'teks_stemmed', 'label']]
df_clean = df_selected.dropna(subset=['teks_stemmed'])
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14305 entries, 0 to 14998
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   reviewId      14305 non-null  object
 1   teks_stemmed  14305 non-null  object
 2   label         14305 non-null  object
dtypes: object(3)
memory usage: 447.0+ KB


In [ ]:
X = df_clean['teks_stemmed'].tolist()
y = df_clean['label']

# Feature Extraction: Word2Vec

Tokenisasi teks dengan memisahkan setiap kata berdasarkan spasi:

In [ ]:
X_tokenized = [teks.split() for teks in X]

Latih model Word2Vec dengan parameter yang ditentukan:

In [ ]:
word2vec_model = Word2Vec(
    sentences=X_tokenized,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)

Buat fungsi untuk mengubah token dokumen menjadi satu vektor fitur:

In [ ]:
def get_document_vector(tokens, model, vector_size=100):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

Terapkan fungsi ke seluruh dokumen dan cetak dimensi data hasil ekstraksi fitur:

In [ ]:
X_word2vec = np.array([get_document_vector(tokens, word2vec_model, 100) for tokens in X_tokenized])
print(f"Shape data Word2Vec: {X_word2vec.shape}")

Shape data Word2Vec: (14305, 100)


Bagi data menjadi data latih (80%) untuk melatih model dan data uji (20%) untuk mengevaluasi kinerja model:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_word2vec, y, test_size=0.2, random_state=42
)

## Model SVM

Fungsi objective untuk optimasi hyperparameter SVM dengan Optuna:

In [ ]:
def objective_svm(trial):
    C = trial.suggest_float('C', 0.01, 100, log=True)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

    svm = SVC(C=C, kernel=kernel, gamma=gamma, random_state=42)
    scores = cross_val_score(svm, X_train, y_train, cv=3, scoring='accuracy', n_jobs=-1)

    return scores.mean()

Mencari hyperparameter terbaik:

In [ ]:
study_svm = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_svm.optimize(objective_svm, n_trials=15, show_progress_bar=True)

[I 2026-04-21 01:36:44,042] A new study created in memory with name: no-name-a90b23f3-2b6b-4bf0-8c84-348692c63cee


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-04-21 01:36:54,801] Trial 0 finished with value: 0.8418382712239724 and parameters: {'C': 0.31489116479568624, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 0 with value: 0.8418382712239724.
[I 2026-04-21 01:37:05,695] Trial 1 finished with value: 0.7732436863749315 and parameters: {'C': 0.042070539502879395, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.8418382712239724.
[I 2026-04-21 01:37:16,496] Trial 2 finished with value: 0.8355466730261668 and parameters: {'C': 0.012087541473056965, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 0 with value: 0.8418382712239724.
[I 2026-04-21 01:37:30,021] Trial 3 finished with value: 0.8414884987204255 and parameters: {'C': 0.05415244119402541, 'kernel': 'rbf', 'gamma': 'scale'}. Best is trial 0 with value: 0.8418382712239724.
[I 2026-04-21 01:37:39,954] Trial 4 finished with value: 0.8427120381258444 and parameters: {'C': 2.801635158716261, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 4 with value: 0

In [ ]:
print("Parameter terbaik:")
for key, value in study_svm.best_params.items():
    print(f"  • {key}: {value}")
print(f"\nAkurasi CV terbaik: {study_svm.best_value:.4f}")

Parameter terbaik:
  • C: 26.336142555139954
  • kernel: rbf
  • gamma: scale

Akurasi CV terbaik: 0.8494


Latih model SVM terbaik dengan seluruh data training:

In [ ]:
svm_best = SVC(**study_svm.best_params, random_state=42)
svm_best.fit(X_train, y_train)

SVC(C=26.336142555139954, random_state=42)

Evaluasi model pada data train dan data test:

In [ ]:
y_pred_train_svm_w2v = svm_best.predict(X_train)
akurasi_train_svm_w2v = accuracy_score(y_train, y_pred_train_svm_w2v)

In [ ]:
y_pred_test_svm_w2v = svm_best.predict(X_test)
akurasi_test_svm_w2v = accuracy_score(y_test, y_pred_test_svm_w2v)

In [ ]:
print(f"Akurasi di data train: {akurasi_train_svm_w2v:.4f}")
print(f"Akurasi di data test: {akurasi_test_svm_w2v:.4f}")
print(f"\nLaporan Klasifikasi:")
print(classification_report(y_test, y_pred_test_svm_w2v))

Akurasi di data train: 0.8591
Akurasi di data test: 0.8556

Laporan Klasifikasi:
              precision    recall  f1-score   support

    negative       0.69      0.71      0.70       537
     neutral       0.00      0.00      0.00       128
    positive       0.90      0.94      0.92      2196

    accuracy                           0.86      2861
   macro avg       0.53      0.55      0.54      2861
weighted avg       0.82      0.86      0.84      2861



Akurasi pada data test mencapai 85.6%

## RNN

Import library tambahan:

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

Encode label menjadi numerik untuk RNN:

In [ ]:
le = LabelEncoder()
y_train_rnn = le.fit_transform(y_train)
y_test_rnn = le.transform(y_test)

Reshape data untuk input RNN:

In [ ]:
X_train_rnn = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_rnn = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

Definisikan fungsi objective untuk optimasi hyperparameter RNN:

In [ ]:
def objective_rnn(trial):
    units = trial.suggest_int('units', 16, 64)
    dropout_rate = trial.suggest_float('dropout', 0.1, 0.5)
    activation = trial.suggest_categorical('activation', ['relu', 'tanh'])
    batch_size = trial.suggest_categorical('batch_size', [32, 64])

    model = Sequential([
        SimpleRNN(units, activation=activation, input_shape=(1, X_train.shape[1])),
        Dropout(dropout_rate),
        Dense(16, activation='relu'),
        Dense(len(le.classes_), activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    history = model.fit(
        X_train_rnn, y_train_rnn,
        epochs=8,
        batch_size=batch_size,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=0
    )

    val_acc = max(history.history['val_accuracy'])
    return val_acc

Optimasi hyperparameter RNN:

In [ ]:
study_rnn = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_rnn.optimize(objective_rnn, n_trials=15, show_progress_bar=True)

[I 2026-04-21 01:44:38,336] A new study created in memory with name: no-name-ed555539-e644-409a-8326-6922c9f52584


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-04-21 01:44:54,432] Trial 0 finished with value: 0.843599796295166 and parameters: {'units': 34, 'dropout': 0.4802857225639665, 'activation': 'relu', 'batch_size': 32}. Best is trial 0 with value: 0.843599796295166.
[I 2026-04-21 01:45:04,078] Trial 1 finished with value: 0.8409786224365234 and parameters: {'units': 18, 'dropout': 0.4464704583099741, 'activation': 'tanh', 'batch_size': 64}. Best is trial 0 with value: 0.843599796295166.
[I 2026-04-21 01:45:13,004] Trial 2 finished with value: 0.840104877948761 and parameters: {'units': 56, 'dropout': 0.18493564427131048, 'activation': 'tanh', 'batch_size': 64}. Best is trial 0 with value: 0.843599796295166.
[I 2026-04-21 01:45:21,801] Trial 3 finished with value: 0.8409786224365234 and parameters: {'units': 37, 'dropout': 0.21649165607921678, 'activation': 'relu', 'batch_size': 64}. Best is trial 0 with value: 0.843599796295166.
[I 2026-04-21 01:45:34,414] Trial 4 finished with value: 0.8409786224365234 and parameters: {'units'

In [ ]:
print("Parameter terbaik:")
for key, value in study_rnn.best_params.items():
    print(f"  • {key}: {value}")
print(f"\nAkurasi CV terbaik: {study_rnn.best_value:.4f}")

Parameter terbaik:
  • units: 30
  • dropout: 0.26486927246395525
  • activation: relu
  • batch_size: 32

Akurasi CV terbaik: 0.8440


Bangun model RNN dengan hyperparameter terbaik:

In [ ]:
best_params = study_rnn.best_params

model_rnn = Sequential([
    SimpleRNN(best_params['units'], activation=best_params['activation'], input_shape=(1, X_train.shape[1])),
    Dropout(best_params['dropout']),
    Dense(16, activation='relu'),
    Dense(len(le.classes_), activation='softmax')
])

model_rnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_rnn = model_rnn.fit(
    X_train_rnn, y_train_rnn,
    epochs=20,
    batch_size=best_params['batch_size'],
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8027 - loss: 0.5333 - val_accuracy: 0.8344 - val_loss: 0.4581
Epoch 2/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8344 - loss: 0.4465 - val_accuracy: 0.8344 - val_loss: 0.4510
Epoch 3/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8399 - loss: 0.4369 - val_accuracy: 0.8379 - val_loss: 0.4404
Epoch 4/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8401 - loss: 0.4318 - val_accuracy: 0.8384 - val_loss: 0.4417
Epoch 5/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8435 - loss: 0.4269 - val_accuracy: 0.8419 - val_loss: 0.4400
Epoch 6/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8390 - loss: 0.4262 - val_accuracy: 0.8401 - val_loss: 0.4364
Epoch 7/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8411 - loss: 0.4251 - val_accuracy: 0.8392 - val_loss: 0.4368
Epoch 8/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8431 - loss: 0.4222 - val_accuracy: 0.

Prediksi dan evaluasi pada data train dan test:

In [ ]:
y_pred_train_rnn_w2v = model_rnn.predict(X_train_rnn)
y_pred_train_class_rnn_w2v = np.argmax(y_pred_train_rnn_w2v, axis=1)
akurasi_train_rnn_w2v = accuracy_score(y_train_rnn, y_pred_train_class_rnn_w2v)

358/358 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step


In [ ]:
y_pred_test_rnn_w2v = model_rnn.predict(X_test_rnn)
y_pred_class_test_rnn_w2v = np.argmax(y_pred_test_rnn_w2v, axis=1)
akurasi_test_rnn_w2v = accuracy_score(y_test_rnn, y_pred_class_test_rnn_w2v)

90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [ ]:
print(f"Akurasi di data train: {akurasi_train_rnn_w2v:.4f}")
print(f"Akurasi di data test: {akurasi_test_rnn_w2v:.4f}")
print(f"\nLaporan Klasifikasi:")
print(classification_report(y_test_rnn, y_pred_class_test_rnn_w2v, target_names=le.classes_))

Akurasi di data train: 0.8473
Akurasi di data test: 0.8511

Laporan Klasifikasi:
              precision    recall  f1-score   support

    negative       0.65      0.74      0.69       537
     neutral       0.00      0.00      0.00       128
    positive       0.90      0.93      0.92      2196

    accuracy                           0.85      2861
   macro avg       0.52      0.56      0.54      2861
weighted avg       0.82      0.85      0.83      2861



Akurasi pada data test mencapai 85.1%



## LSTM

Import library tambahan:

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding

Encode label menjadi numerik untuk LSTM:

In [ ]:
le = LabelEncoder()
y_train_lstm = le.fit_transform(y_train)
y_test_lstm = le.transform(y_test)

Reshape data:

In [ ]:
X_train_lstm = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_lstm = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

Fungsi objective untuk optimasi hyperparameter LSTM:

In [ ]:
def objective_lstm(trial):
    units = trial.suggest_int('units', 16, 48)
    dropout_rate = trial.suggest_float('dropout', 0.2, 0.4)
    batch_size = trial.suggest_categorical('batch_size', [32, 64])

    model = Sequential([
        LSTM(units, activation='tanh', input_shape=(1, X_train.shape[1])),
        Dropout(dropout_rate),
        Dense(16, activation='relu'),
        Dense(len(le.classes_), activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    history = model.fit(
        X_train_lstm, y_train_lstm,
        epochs=8,
        batch_size=batch_size,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=0
    )

    val_acc = max(history.history['val_accuracy'])
    return val_acc

Optimasi hyperparameter LSTM:

In [ ]:
study_lstm = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_lstm.optimize(objective_lstm, n_trials=15, show_progress_bar=True)

[I 2026-04-21 01:50:58,194] A new study created in memory with name: no-name-722567af-b86e-4784-a00a-f6a9245c8a66


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-04-21 01:51:24,754] Trial 0 finished with value: 0.8409786224365234 and parameters: {'units': 28, 'dropout': 0.3901428612819833, 'batch_size': 32}. Best is trial 0 with value: 0.8409786224365234.
[I 2026-04-21 01:51:47,763] Trial 1 finished with value: 0.8409786224365234 and parameters: {'units': 21, 'dropout': 0.23119890406724053, 'batch_size': 64}. Best is trial 0 with value: 0.8409786224365234.
[I 2026-04-21 01:52:12,139] Trial 2 finished with value: 0.8405417203903198 and parameters: {'units': 35, 'dropout': 0.3416145155592091, 'batch_size': 64}. Best is trial 0 with value: 0.8409786224365234.
[I 2026-04-21 01:52:29,185] Trial 3 finished with value: 0.8418523669242859 and parameters: {'units': 43, 'dropout': 0.24246782213565524, 'batch_size': 64}. Best is trial 3 with value: 0.8418523669242859.
[I 2026-04-21 01:52:53,875] Trial 4 finished with value: 0.8387942314147949 and parameters: {'units': 26, 'dropout': 0.3049512863264476, 'batch_size': 32}. Best is trial 3 with value

In [ ]:
print("Parameter terbaik:")
for key, value in study_lstm.best_params.items():
    print(f"  • {key}: {value}")
print(f"\nAkurasi CV terbaik: {study_lstm.best_value:.4f}")

Parameter terbaik:
  • units: 43
  • dropout: 0.24246782213565524
  • batch_size: 64

Akurasi CV terbaik: 0.8419


Bangun model LSTM dengan hyperparameter terbaik:

In [ ]:
best_params = study_lstm.best_params

model_lstm = Sequential([
    LSTM(best_params['units'], activation='tanh', input_shape=(1, X_train.shape[1])),
    Dropout(best_params['dropout']),
    Dense(16, activation='relu'),
    Dense(len(le.classes_), activation='softmax')
])

model_lstm.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_lstm = model_lstm.fit(
    X_train_lstm, y_train_lstm,
    epochs=20,
    batch_size=best_params['batch_size'],
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.7961 - loss: 0.5762 - val_accuracy: 0.8322 - val_loss: 0.4720
Epoch 2/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8330 - loss: 0.4577 - val_accuracy: 0.8322 - val_loss: 0.4565
Epoch 3/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8351 - loss: 0.4453 - val_accuracy: 0.8353 - val_loss: 0.4482
Epoch 4/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8426 - loss: 0.4339 - val_accuracy: 0.8401 - val_loss: 0.4433
Epoch 5/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8402 - loss: 0.4291 - val_accuracy: 0.8388 - val_loss: 0.4421
Epoch 6/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8430 - loss: 0.4253 - val_accuracy: 0.8384 - val_loss: 0.4423
Epoch 7/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8426 - loss: 0.4226 - val_accuracy: 0.8401 - val_loss: 0.4351
Epoch 8/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8421 - loss: 0.4188 - val_accuracy: 0

Prediksi dan evaluasi pada data train dan test:

In [ ]:
y_pred_train_lstm_w2v = model_lstm.predict(X_train_lstm)
y_pred_train_class_lstm_w2v = np.argmax(y_pred_train_lstm_w2v, axis=1)
akurasi_train_lstm_w2v = accuracy_score(y_train_lstm, y_pred_train_class_lstm_w2v)

358/358 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [ ]:
y_pred_test_lstm_w2v = model_lstm.predict(X_test_lstm)
y_pred_test_class_lstm_w2v = np.argmax(y_pred_test_lstm_w2v, axis=1)
akurasi_test_lstm_w2v = accuracy_score(y_test_lstm, y_pred_test_class_lstm_w2v)

90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [ ]:
print(f"Akurasi di data train: {akurasi_train_lstm_w2v:.4f}")
print(f"Akurasi di data test: {akurasi_test_lstm_w2v:.4f}")
print(f"\nLaporan Klasifikasi:")
print(classification_report(y_test_lstm, y_pred_test_class_lstm_w2v, target_names=le.classes_))

Akurasi di data train: 0.8466
Akurasi di data test: 0.8504

Laporan Klasifikasi:
              precision    recall  f1-score   support

    negative       0.68      0.68      0.68       537
     neutral       0.00      0.00      0.00       128
    positive       0.89      0.94      0.92      2196

    accuracy                           0.85      2861
   macro avg       0.52      0.54      0.53      2861
weighted avg       0.81      0.85      0.83      2861



Akurasi pada data test mencapai 85%

# Feature Extraction: BOW

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

Inisialisasi CountVectorizer untuk ekstraksi fitur Bag of Words:

In [ ]:
bow = CountVectorizer(max_features=1000)
X_bow = bow.fit_transform(X)

Split data menjadi train-test dengan rasio 80:20:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_bow, y, test_size=0.2, random_state=42, stratify=y
)

Encode label menjadi numerik untuk model machine learning:

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

Konversi matriks sparse BoW menjadi dense array:

In [ ]:
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

Reshape data untuk input RNN:

In [ ]:
X_train_rnn = X_train_dense.reshape(X_train_dense.shape[0], 1, X_train_dense.shape[1])
X_test_rnn = X_test_dense.reshape(X_test_dense.shape[0], 1, X_test_dense.shape[1])

## SVM

Fungsi objective untuk optimasi hyperparameter SVM dengan Optuna:

In [ ]:
def objective_svm(trial):
    C = trial.suggest_float('C', 0.01, 100, log=True)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

    svm = SVC(C=C, kernel=kernel, gamma=gamma, random_state=42)
    scores = cross_val_score(svm, X_train, y_train, cv=3, scoring='accuracy', n_jobs=-1)
    return scores.mean()

Mencari hyperparameter terbaik:

In [ ]:
study_svm_bow = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_svm_bow.optimize(objective_svm, n_trials=15, show_progress_bar=True)

[I 2026-04-21 02:01:25,095] A new study created in memory with name: no-name-6db8449b-6d76-4dd4-98ef-803983d80472


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-04-21 02:01:33,389] Trial 0 finished with value: 0.8493528819233731 and parameters: {'C': 0.31489116479568624, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 0 with value: 0.8493528819233731.
[I 2026-04-21 02:01:41,630] Trial 1 finished with value: 0.7695735721536369 and parameters: {'C': 0.042070539502879395, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.8493528819233731.
[I 2026-04-21 02:01:54,198] Trial 2 finished with value: 0.8346727228533997 and parameters: {'C': 0.012087541473056965, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 0 with value: 0.8493528819233731.
[I 2026-04-21 02:02:12,651] Trial 3 finished with value: 0.8278569698952355 and parameters: {'C': 0.05415244119402541, 'kernel': 'rbf', 'gamma': 'scale'}. Best is trial 0 with value: 0.8493528819233731.
[I 2026-04-21 02:02:24,641] Trial 4 finished with value: 0.825323181042092 and parameters: {'C': 2.801635158716261, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.

In [ ]:
print("\nParameter terbaik SVM:", study_svm_bow.best_params)
print(f"Akurasi CV terbaik: {study_svm_bow.best_value:.4f}")


Parameter terbaik SVM: {'C': 86.69626877293648, 'kernel': 'rbf', 'gamma': 'auto'}
Akurasi CV terbaik: 0.8510


Latih model SVM terbaik dengan seluruh data training BoW:

In [ ]:
svm_best_bow = SVC(**study_svm_bow.best_params, random_state=42)
svm_best_bow.fit(X_train, y_train)

Evaluasi model pada data train dan data test:

In [ ]:
y_pred_test_svm_bow = svm_best_bow.predict(X_test)
acc_test_svm_bow = accuracy_score(y_test, y_pred_test_svm_bow)

In [ ]:
y_pred_train_svm_bow = svm_best_bow.predict(X_train)
acc_train_svm_bow = accuracy_score(y_train, y_pred_train_svm_bow)

In [ ]:
print(f"Akurasi DATA TRAIN: {acc_train_svm_bow:.4f}")
print(f"Akurasi DATA TEST : {acc_test_svm_bow:.4f}")
print(classification_report(y_test, y_pred_test_svm_bow))

Akurasi DATA TRAIN: 0.8796
Akurasi DATA TEST : 0.8581
              precision    recall  f1-score   support

    negative       0.78      0.59      0.67       542
     neutral       0.25      0.01      0.02       117
    positive       0.87      0.97      0.92      2202

    accuracy                           0.86      2861
   macro avg       0.63      0.52      0.54      2861
weighted avg       0.83      0.86      0.83      2861



Akurasi pada data test mencapai 85.8%

## RNN

Definisikan fungsi objective untuk optimasi hyperparameter RNN dengan data BoW:

In [ ]:
def objective_rnn(trial):
    units = trial.suggest_int('units', 16, 64)
    dropout_rate = trial.suggest_float('dropout', 0.1, 0.5)
    activation = trial.suggest_categorical('activation', ['relu', 'tanh'])
    batch_size = trial.suggest_categorical('batch_size', [32, 64])

    model = Sequential([
        SimpleRNN(units, activation=activation, input_shape=(1, X_train_dense.shape[1])),
        Dropout(dropout_rate),
        Dense(16, activation='relu'),
        Dense(len(le.classes_), activation='softmax')
    ])

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    history = model.fit(
        X_train_rnn, y_train_enc,
        epochs=8,
        batch_size=batch_size,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=0
    )

    return max(history.history['val_accuracy'])

Hyperparameter tuning:

In [ ]:
study_rnn_bow = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_rnn_bow.optimize(objective_rnn, n_trials=15, show_progress_bar=True)

print("\nParameter terbaik RNN:", study_rnn_bow.best_params)
print(f"Akurasi CV terbaik: {study_rnn_bow.best_value:.4f}")

[I 2026-04-21 02:07:39,689] A new study created in memory with name: no-name-4410cde0-1b8a-4ca8-b647-4d1a22bb8e57


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-04-21 02:07:51,204] Trial 0 finished with value: 0.8680646419525146 and parameters: {'units': 34, 'dropout': 0.4802857225639665, 'activation': 'relu', 'batch_size': 32}. Best is trial 0 with value: 0.8680646419525146.
[I 2026-04-21 02:08:00,819] Trial 1 finished with value: 0.8667540550231934 and parameters: {'units': 18, 'dropout': 0.4464704583099741, 'activation': 'tanh', 'batch_size': 64}. Best is trial 0 with value: 0.8680646419525146.
[I 2026-04-21 02:08:09,348] Trial 2 finished with value: 0.8628221750259399 and parameters: {'units': 56, 'dropout': 0.18493564427131048, 'activation': 'tanh', 'batch_size': 64}. Best is trial 0 with value: 0.8680646419525146.
[I 2026-04-21 02:08:16,980] Trial 3 finished with value: 0.8663171529769897 and parameters: {'units': 37, 'dropout': 0.21649165607921678, 'activation': 'relu', 'batch_size': 64}. Best is trial 0 with value: 0.8680646419525146.
[I 2026-04-21 02:08:27,130] Trial 4 finished with value: 0.864132821559906 and parameters: {'u

Bangun model RNN dengan hyperparameter terbaik:

In [ ]:
best_params = study_rnn_bow.best_params
model_rnn_bow = Sequential([
    SimpleRNN(best_params['units'], activation=best_params['activation'], input_shape=(1, X_train_dense.shape[1])),
    Dropout(best_params['dropout']),
    Dense(16, activation='relu'),
    Dense(len(le.classes_), activation='softmax')
])

model_rnn_bow.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_rnn = model_rnn_bow.fit(
    X_train_rnn, y_train_enc,
    epochs=20,
    batch_size=best_params['batch_size'],
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.7932 - loss: 0.7238 - val_accuracy: 0.8536 - val_loss: 0.4112
Epoch 2/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8561 - loss: 0.4086 - val_accuracy: 0.8633 - val_loss: 0.3747
Epoch 3/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8726 - loss: 0.3667 - val_accuracy: 0.8646 - val_loss: 0.3704
Epoch 4/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8835 - loss: 0.3415 - val_accuracy: 0.8672 - val_loss: 0.3719
Epoch 5/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8927 - loss: 0.3195 - val_accuracy: 0.8685 - val_loss: 0.3795
Epoch 6/20
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8978 - loss: 0.3028 - val_accuracy: 0.8668 - val_loss: 0.3864


Prediksi dan evaluasi pada data train dan test:

In [ ]:
y_pred_rnn_train_bow = model_rnn_bow.predict(X_train_rnn)
y_pred_rnn_train_class_bow = np.argmax(y_pred_rnn_train_bow, axis=1)
acc_train_rnn_bow = accuracy_score(y_train_enc, y_pred_rnn_train_class_bow)

358/358 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step


In [ ]:
y_pred_rnn_test_bow = model_rnn_bow.predict(X_test_rnn)
y_pred_rnn_test_class_bow = np.argmax(y_pred_rnn_test_bow, axis=1)
acc_test_rnn_bow = accuracy_score(y_test_enc, y_pred_rnn_test_class_bow)

90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step


In [ ]:
print(f"Akurasi DATA TRAIN: {acc_train_rnn_bow:.4f}")
print(f"Akurasi DATA TEST : {acc_test_rnn_bow:.4f}")

Akurasi DATA TRAIN: 0.8825
Akurasi DATA TEST : 0.8661


In [ ]:
print(classification_report(y_test_enc, y_pred_rnn_test_class_bow, target_names=le.classes_))

              precision    recall  f1-score   support

    negative       0.72      0.72      0.72       542
     neutral       0.00      0.00      0.00       117
    positive       0.90      0.95      0.92      2202

    accuracy                           0.87      2861
   macro avg       0.54      0.56      0.55      2861
weighted avg       0.83      0.87      0.85      2861



Akurasi pada data test mencapai 86.6%

## LSTM

Definisikan fungsi objective untuk optimasi hyperparameter LSTM:

In [ ]:
def objective_lstm(trial):
    units = trial.suggest_int('units', 16, 48)
    dropout_rate = trial.suggest_float('dropout', 0.2, 0.4)
    batch_size = trial.suggest_categorical('batch_size', [32, 64])

    model = Sequential([
        LSTM(units, activation='tanh', input_shape=(1, X_train_dense.shape[1])),
        Dropout(dropout_rate),
        Dense(16, activation='relu'),
        Dense(len(le.classes_), activation='softmax')
    ])

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    history = model.fit(
        X_train_rnn, y_train_enc,
        epochs=8,
        batch_size=batch_size,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=0
    )

    return max(history.history['val_accuracy'])

Hyperparameter Tuning:

In [ ]:
study_lstm_bow = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lstm_bow.optimize(objective_lstm, n_trials=15, show_progress_bar=True)

print("\nParameter terbaik LSTM:", study_lstm_bow.best_params)
print(f"Akurasi CV terbaik: {study_lstm_bow.best_value:.4f}")

[I 2026-04-21 02:11:49,133] A new study created in memory with name: no-name-fb2dc250-15f7-4102-8f54-08e7be5d2429


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-04-21 02:12:00,674] Trial 0 finished with value: 0.8654434084892273 and parameters: {'units': 28, 'dropout': 0.3901428612819833, 'batch_size': 32}. Best is trial 0 with value: 0.8654434084892273.
[I 2026-04-21 02:12:08,376] Trial 1 finished with value: 0.8628221750259399 and parameters: {'units': 21, 'dropout': 0.23119890406724053, 'batch_size': 64}. Best is trial 0 with value: 0.8654434084892273.
[I 2026-04-21 02:12:17,158] Trial 2 finished with value: 0.8667540550231934 and parameters: {'units': 35, 'dropout': 0.3416145155592091, 'batch_size': 64}. Best is trial 2 with value: 0.8667540550231934.
[I 2026-04-21 02:12:24,402] Trial 3 finished with value: 0.8671908974647522 and parameters: {'units': 43, 'dropout': 0.24246782213565524, 'batch_size': 64}. Best is trial 3 with value: 0.8671908974647522.
[I 2026-04-21 02:12:35,725] Trial 4 finished with value: 0.8680646419525146 and parameters: {'units': 26, 'dropout': 0.3049512863264476, 'batch_size': 32}. Best is trial 4 with value

Bangun model LSTM dengan hyperparameter terbaik:

In [ ]:
best_params = study_lstm_bow.best_params
model_lstm_bow = Sequential([
    LSTM(best_params['units'], activation='tanh', input_shape=(1, X_train_dense.shape[1])),
    Dropout(best_params['dropout']),
    Dense(16, activation='relu'),
    Dense(len(le.classes_), activation='softmax')
])

model_lstm_bow.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_lstm = model_lstm_bow.fit(
    X_train_rnn, y_train_enc,
    epochs=20,
    batch_size=best_params['batch_size'],
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8057 - loss: 0.6816 - val_accuracy: 0.8475 - val_loss: 0.4071
Epoch 2/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8532 - loss: 0.4148 - val_accuracy: 0.8580 - val_loss: 0.3779
Epoch 3/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8632 - loss: 0.3850 - val_accuracy: 0.8637 - val_loss: 0.3694
Epoch 4/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.8726 - loss: 0.3633 - val_accuracy: 0.8654 - val_loss: 0.3693
Epoch 5/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8783 - loss: 0.3496 - val_accuracy: 0.8659 - val_loss: 0.3747
Epoch 6/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8842 - loss: 0.3380 - val_accuracy: 0.8663 - val_loss: 0.3789
Epoch 7/20
287/287 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.8885 - loss: 0.3256 - val_accuracy: 0.8646 - val_loss: 0.3876


Prediksi dan evaluasi pada data train dan test:

In [ ]:
y_pred_lstm_train_bow = model_lstm_bow.predict(X_train_rnn)
y_pred_lstm_train_class_bow = np.argmax(y_pred_lstm_train_bow, axis=1)
acc_lstm_train_bow = accuracy_score(y_train_enc, y_pred_lstm_train_class_bow)

358/358 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [ ]:
y_pred_lstm_test_bow = model_lstm_bow.predict(X_test_rnn)
y_pred_lstm_test_class_bow = np.argmax(y_pred_lstm_test_bow, axis=1)
acc_test_lstm_bow = accuracy_score(y_test_enc, y_pred_lstm_test_class_bow)

90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [ ]:
print(f"Akurasi DATA TRAIN: {acc_lstm_train_bow:.4f}")
print(f"Akurasi DATA TEST : {acc_test_lstm_bow:.4f}")

Akurasi DATA TRAIN: 0.8791
Akurasi DATA TEST : 0.8672


In [ ]:
print(classification_report(y_test_enc, y_pred_lstm_test_class_bow, target_names=le.classes_))

              precision    recall  f1-score   support

    negative       0.74      0.70      0.72       542
     neutral       0.00      0.00      0.00       117
    positive       0.90      0.95      0.92      2202

    accuracy                           0.87      2861
   macro avg       0.54      0.55      0.55      2861
weighted avg       0.83      0.87      0.85      2861



Akurasi pada data test mencapai 86.7%

# Feature Extraction: TF-IDF

Import library tambahan:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

Inisialisasi TfidfVectorizer:

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 1),
    lowercase=True,
    min_df=2,
    max_df=0.95
)

X_tfidf = tfidf_vectorizer.fit_transform(X)
print(f"Shape TF-IDF matrix: {X_tfidf.shape}")
print(f"Jumlah fitur: {X_tfidf.shape[1]}")

Shape TF-IDF matrix: (14305, 1918)
Jumlah fitur: 1918


Split data menjadi train-test dengan rasio 80:20:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## SVM

Definisikan fungsi objective untuk optimasi hyperparameter SVM dengan data TF-IDF:

In [ ]:
def objective_svm(trial):
    C = trial.suggest_float('C', 0.01, 100, log=True)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

    svm = SVC(C=C, kernel=kernel, gamma=gamma, random_state=42)
    scores = cross_val_score(svm, X_train, y_train, cv=3, scoring='accuracy', n_jobs=-1)

    return scores.mean()

Hyperparameter tuning:

In [ ]:
study_svm_tfidf = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_svm_tfidf.optimize(objective_svm, n_trials=15, show_progress_bar=True)

[I 2026-04-21 02:30:19,895] A new study created in memory with name: no-name-e2f35604-664c-44d1-8af4-36d1f6bc8350


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-04-21 02:30:28,949] Trial 0 finished with value: 0.8509260334703055 and parameters: {'C': 0.31489116479568624, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 0 with value: 0.8509260334703055.
[I 2026-04-21 02:30:38,348] Trial 1 finished with value: 0.7695735721536369 and parameters: {'C': 0.042070539502879395, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.8509260334703055.
[I 2026-04-21 02:30:46,514] Trial 2 finished with value: 0.7695735721536369 and parameters: {'C': 0.012087541473056965, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 0 with value: 0.8509260334703055.
[I 2026-04-21 02:30:57,149] Trial 3 finished with value: 0.7700978872760288 and parameters: {'C': 0.05415244119402541, 'kernel': 'rbf', 'gamma': 'scale'}. Best is trial 0 with value: 0.8509260334703055.
[I 2026-04-21 02:31:05,078] Trial 4 finished with value: 0.7695735721536369 and parameters: {'C': 2.801635158716261, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0

In [ ]:
print("Parameter terbaik:")
for key, value in study_svm_tfidf.best_params.items():
    print(f"  • {key}: {value}")
print(f"\nAkurasi CV terbaik: {study_svm_tfidf.best_value:.4f}")

Parameter terbaik:
  • C: 2.139185846013699
  • kernel: rbf
  • gamma: scale

Akurasi CV terbaik: 0.8580


Latih model SVM terbaik:

In [ ]:
svm_best_tfidf = SVC(**study_svm_tfidf.best_params, random_state=42)
svm_best_tfidf.fit(X_train, y_train)

SVC(C=2.139185846013699, random_state=42)

Prediksi dan evaluasi pada data test dan train:

In [ ]:
y_pred_test_tfidf = svm_best_tfidf.predict(X_test)
akurasi_test_tfidf = accuracy_score(y_test, y_pred_test_tfidf)

y_pred_train_tfidf = svm_best_tfidf.predict(X_train)
akurasi_train_tfidf = accuracy_score(y_train, y_pred_train_tfidf)

In [ ]:
print(f"Akurasi di data train: {akurasi_train_tfidf:.4f}")
print(f"Akurasi di data test: {akurasi_test_tfidf:.4f}")
print(f"\nLaporan Klasifikasi:")
print(classification_report(y_test, y_pred_test_tfidf))

Akurasi di data train: 0.9665
Akurasi di data test: 0.8630

Laporan Klasifikasi:
              precision    recall  f1-score   support

    negative       0.73      0.69      0.71       542
     neutral       0.00      0.00      0.00       117
    positive       0.89      0.95      0.92      2202

    accuracy                           0.86      2861
   macro avg       0.54      0.55      0.54      2861
weighted avg       0.83      0.86      0.84      2861



Akurasi pada data test mencapai 86.3%

## RNN

Encode label menjadi numerik untuk RNN:

In [ ]:
le = LabelEncoder()
y_train_rnn = le.fit_transform(y_train)
y_test_rnn = le.transform(y_test)

In [ ]:
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()
X_train_rnn = X_train_dense.reshape(X_train_dense.shape[0], 1, X_train_dense.shape[1])
X_test_rnn = X_test_dense.reshape(X_test_dense.shape[0], 1, X_test_dense.shape[1])

Fungsi objective untuk optimasi hyperparameter RNN:

In [ ]:
def objective_rnn(trial):
    units = trial.suggest_int('units', 16, 64)
    dropout_rate = trial.suggest_float('dropout', 0.1, 0.5)
    activation = trial.suggest_categorical('activation', ['relu', 'tanh'])
    batch_size = trial.suggest_categorical('batch_size', [32, 64])

    model = Sequential([
        SimpleRNN(units, activation=activation, input_shape=(1, X_train_rnn.shape[2])),
        Dropout(dropout_rate),
        Dense(16, activation='relu'),
        Dense(len(le.classes_), activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    history = model.fit(
        X_train_rnn, y_train_rnn,
        epochs=8,
        batch_size=batch_size,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=0
    )

    val_acc = max(history.history['val_accuracy'])
    return val_acc

Hyperparameter tuning:

In [ ]:
study_rnn_tfidf = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_rnn_tfidf.optimize(objective_rnn, n_trials=15, show_progress_bar=True)

[I 2026-04-21 02:33:47,366] A new study created in memory with name: no-name-651604fb-5b22-45e8-af21-2d8fcfef5b71


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-04-21 02:33:58,483] Trial 0 finished with value: 0.8685015439987183 and parameters: {'units': 34, 'dropout': 0.4802857225639665, 'activation': 'relu', 'batch_size': 32}. Best is trial 0 with value: 0.8685015439987183.
[I 2026-04-21 02:34:06,620] Trial 1 finished with value: 0.8698121309280396 and parameters: {'units': 18, 'dropout': 0.4464704583099741, 'activation': 'tanh', 'batch_size': 64}. Best is trial 1 with value: 0.8698121309280396.
[I 2026-04-21 02:34:15,098] Trial 2 finished with value: 0.8676277995109558 and parameters: {'units': 56, 'dropout': 0.18493564427131048, 'activation': 'tanh', 'batch_size': 64}. Best is trial 1 with value: 0.8698121309280396.
[I 2026-04-21 02:34:24,046] Trial 3 finished with value: 0.8663171529769897 and parameters: {'units': 37, 'dropout': 0.21649165607921678, 'activation': 'relu', 'batch_size': 64}. Best is trial 1 with value: 0.8698121309280396.
[I 2026-04-21 02:34:34,111] Trial 4 finished with value: 0.8689383864402771 and parameters: {'

In [ ]:
print("Parameter terbaik:")
for key, value in study_rnn_tfidf.best_params.items():
    print(f"  • {key}: {value}")
print(f"\nAkurasi validasi terbaik: {study_rnn_tfidf.best_value:.4f}")

Parameter terbaik:
  • units: 45
  • dropout: 0.4687496940092467
  • activation: tanh
  • batch_size: 64

Akurasi validasi terbaik: 0.8729


Bangun model RNN dengan hyperparameter terbaik:

In [ ]:
best_params = study_rnn_tfidf.best_params

model_rnn_tfidf = Sequential([
    SimpleRNN(best_params['units'], activation=best_params['activation'], input_shape=(1, X_train_rnn.shape[2])),
    Dropout(best_params['dropout']),
    Dense(16, activation='relu'),
    Dense(len(le.classes_), activation='softmax')
])

model_rnn_tfidf.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_final = model_rnn_tfidf.fit(
    X_train_rnn, y_train_rnn,
    epochs=15,
    batch_size=best_params['batch_size'],
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.7525 - loss: 0.6780 - val_accuracy: 0.8449 - val_loss: 0.4187
Epoch 2/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8508 - loss: 0.4119 - val_accuracy: 0.8646 - val_loss: 0.3719
Epoch 3/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8700 - loss: 0.3733 - val_accuracy: 0.8672 - val_loss: 0.3655
Epoch 4/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8823 - loss: 0.3478 - val_accuracy: 0.8654 - val_loss: 0.3677
Epoch 5/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8854 - loss: 0.3322 - val_accuracy: 0.8681 - val_loss: 0.3690
Epoch 6/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8906 - loss: 0.3180 - val_accuracy: 0.8654 - val_loss: 0.3787


Prediksi dan evaluasi pada data test dan train:

In [ ]:
y_pred_test_rnn_tfidf = model_rnn_tfidf.predict(X_test_rnn)
y_pred_test_class_rnn_tfidf = np.argmax(y_pred_test_rnn_tfidf, axis=1)
akurasi_test_rnn_tfidf = accuracy_score(y_test_rnn, y_pred_test_class_rnn_tfidf)

90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


In [ ]:
y_pred_train_rnn_tfidf = model_rnn_tfidf.predict(X_train_rnn)
y_pred_train_class_rnn_tfidf = np.argmax(y_pred_train_rnn_tfidf, axis=1)
akurasi_train_rnn_tfidf = accuracy_score(y_train_rnn, y_pred_train_class_rnn_tfidf)

358/358 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [ ]:
print(f"Akurasi DATA TRAIN: {akurasi_train_rnn_tfidf:.4f}")
print(f"Akurasi DATA TEST : {akurasi_test_rnn_tfidf:.4f}")

Akurasi DATA TRAIN: 0.8802
Akurasi DATA TEST : 0.8609


In [ ]:
print(classification_report(y_test_rnn, y_pred_test_class_rnn_tfidf, target_names=le.classes_))

              precision    recall  f1-score   support

    negative       0.73      0.67      0.70       542
     neutral       0.00      0.00      0.00       117
    positive       0.89      0.95      0.92      2202

    accuracy                           0.86      2861
   macro avg       0.54      0.54      0.54      2861
weighted avg       0.82      0.86      0.84      2861



Akurasi pada data test mencapai 86.1%

## LSTM

In [ ]:
import tensorflow as tf

In [ ]:
le = LabelEncoder()
y_train_lstm = le.fit_transform(y_train)
y_test_lstm = le.transform(y_test)

Konversi matriks sparse TF-IDF ke dense array dan reshape untuk input LSTM (3D format):

In [ ]:
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()
X_train_lstm = X_train_dense.reshape(X_train_dense.shape[0], 1, X_train_dense.shape[1])
X_test_lstm = X_test_dense.reshape(X_test_dense.shape[0], 1, X_test_dense.shape[1])

Definisikan fungsi objective untuk optimasi hyperparameter LSTM:

In [ ]:
def objective_lstm(trial):
    units = trial.suggest_int('units', 16, 48)
    dropout_rate = trial.suggest_float('dropout', 0.2, 0.4)
    batch_size = trial.suggest_categorical('batch_size', [32, 64])

    model = Sequential([
        LSTM(units, activation='tanh', input_shape=(1, X_train_lstm.shape[2])),
        Dropout(dropout_rate),
        Dense(16, activation='relu'),
        Dense(len(le.classes_), activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    history = model.fit(
        X_train_lstm, y_train_lstm,
        epochs=8,
        batch_size=batch_size,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=0
    )

    val_acc = max(history.history['val_accuracy'])
    return val_acc

Hyperparameter tuning:

In [ ]:
study_lstm_tfidf = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_lstm_tfidf.optimize(objective_lstm, n_trials=15, show_progress_bar=True)

[I 2026-04-21 02:37:42,012] A new study created in memory with name: no-name-1c87797b-ad9b-44d5-a752-f455f067887b


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-04-21 02:38:07,342] Trial 0 finished with value: 0.8689383864402771 and parameters: {'units': 28, 'dropout': 0.3901428612819833, 'batch_size': 32}. Best is trial 0 with value: 0.8689383864402771.
[I 2026-04-21 02:38:26,604] Trial 1 finished with value: 0.8719965219497681 and parameters: {'units': 21, 'dropout': 0.23119890406724053, 'batch_size': 64}. Best is trial 1 with value: 0.8719965219497681.
[I 2026-04-21 02:38:47,726] Trial 2 finished with value: 0.870685875415802 and parameters: {'units': 35, 'dropout': 0.3416145155592091, 'batch_size': 64}. Best is trial 1 with value: 0.8719965219497681.
[I 2026-04-21 02:39:05,544] Trial 3 finished with value: 0.8702490329742432 and parameters: {'units': 43, 'dropout': 0.24246782213565524, 'batch_size': 64}. Best is trial 1 with value: 0.8719965219497681.
[I 2026-04-21 02:39:26,249] Trial 4 finished with value: 0.8685015439987183 and parameters: {'units': 26, 'dropout': 0.3049512863264476, 'batch_size': 32}. Best is trial 1 with value:

In [ ]:
print("Parameter terbaik:")
for key, value in study_lstm_tfidf.best_params.items():
    print(f"  • {key}: {value}")
print(f"\nAkurasi validasi terbaik: {study_lstm_tfidf.best_value:.4f}")

Parameter terbaik:
  • units: 21
  • dropout: 0.3930226345779123
  • batch_size: 64

Akurasi validasi terbaik: 0.8729


Bangun model LSTM dengan hyperparameter terbaik:

In [ ]:
best_params = study_lstm_tfidf.best_params

model_lstm_tfidf = Sequential([
    LSTM(best_params['units'], activation='tanh', input_shape=(1, X_train_lstm.shape[2])),
    Dropout(best_params['dropout']),
    Dense(16, activation='relu'),
    Dense(len(le.classes_), activation='softmax')
])

model_lstm_tfidf.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [ ]:
history_final = model_lstm_tfidf.fit(
    X_train_lstm, y_train_lstm,
    epochs=15,
    batch_size=best_params['batch_size'],
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.7526 - loss: 0.8349 - val_accuracy: 0.7798 - val_loss: 0.5360
Epoch 2/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8165 - loss: 0.4674 - val_accuracy: 0.8493 - val_loss: 0.3856
Epoch 3/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8594 - loss: 0.3982 - val_accuracy: 0.8668 - val_loss: 0.3689
Epoch 4/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8700 - loss: 0.3739 - val_accuracy: 0.8698 - val_loss: 0.3647
Epoch 5/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8785 - loss: 0.3575 - val_accuracy: 0.8737 - val_loss: 0.3639
Epoch 6/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8854 - loss: 0.3468 - val_accuracy: 0.8698 - val_loss: 0.3667
Epoch 7/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8890 - loss: 0.3326 - val_accuracy: 0.8641 - val_loss: 0.3706
Epoch 8/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8931 - loss: 0.3234 - val_accurac

Prediksi dan evaluasi pada data test dan train:

In [ ]:
y_pred_test_lstm_tfidf = model_lstm_tfidf.predict(X_test_lstm)
y_pred_test_class_lstm_tfidf = np.argmax(y_pred_test_lstm_tfidf, axis=1)
akurasi_test_lstm_tfidf = accuracy_score(y_test_lstm, y_pred_test_class_lstm_tfidf)

90/90 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


In [ ]:
y_pred_train_lstm_tfidf = model_lstm_tfidf.predict(X_train_lstm)
y_pred_train_class_lstm_tfidf = np.argmax(y_pred_train_lstm_tfidf, axis=1)
akurasi_train_lstm_tfidf = accuracy_score(y_train_lstm, y_pred_train_class_lstm_tfidf)

358/358 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [ ]:
print(f"Akurasi DATA TRAIN: {akurasi_train_lstm_tfidf:.4f}")
print(f"Akurasi DATA TEST : {akurasi_test_lstm_tfidf:.4f}")

Akurasi DATA TRAIN: 0.8849
Akurasi DATA TEST : 0.8637


In [ ]:
print(classification_report(y_test_enc, y_pred_test_class_lstm_tfidf, target_names=le.classes_))

              precision    recall  f1-score   support

    negative       0.72      0.70      0.71       542
     neutral       0.00      0.00      0.00       117
    positive       0.90      0.95      0.92      2202

    accuracy                           0.86      2861
   macro avg       0.54      0.55      0.54      2861
weighted avg       0.83      0.86      0.84      2861



Akurasi pada data test mencapai 86.4%

# Perbandingan Metode

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
akurasi_data = {
    'Feature': ['Word2Vec', 'Word2Vec', 'Word2Vec',
                'BoW', 'BoW', 'BoW',
                'TF-IDF', 'TF-IDF', 'TF-IDF'],
    'Model': ['SVM', 'RNN', 'LSTM',
              'SVM', 'RNN', 'LSTM',
              'SVM', 'RNN', 'LSTM'],
    'Akurasi': [
        # Word2Vec
        akurasi_test_svm_w2v,
        akurasi_test_rnn_w2v,
        akurasi_test_lstm_w2v,
        # BoW
        acc_test_svm_bow,
        acc_test_rnn_bow,
        acc_test_lstm_bow,
        # TF-IDF
        akurasi_test_tfidf,
        akurasi_test_rnn_tfidf,
        akurasi_test_lstm_tfidf
    ]
}

In [ ]:
df_akurasi = pd.DataFrame(akurasi_data)
df_akurasi['Akurasi (%)'] = df_akurasi['Akurasi'].apply(lambda x: f"{x*100:.2f}%")
df_akurasi['Nilai'] = df_akurasi['Akurasi'].apply(lambda x: x*100)

In [ ]:
pivot_table = df_akurasi.pivot(index='Feature', columns='Model', values='Akurasi')
pivot_table_persen = df_akurasi.pivot(index='Feature', columns='Model', values='Akurasi (%)')

In [ ]:
print("TABEL PERBANDINGAN AKURASI MODEL")
print("\nNilai Akurasi (persentase):")
print(pivot_table_persen)

TABEL PERBANDINGAN AKURASI MODEL

Nilai Akurasi (persentase):
Model       LSTM     RNN     SVM
Feature                         
BoW       86.72%  86.61%  85.81%
TF-IDF    86.37%  86.09%  86.30%
Word2Vec  85.04%  85.11%  85.56%


Secara umum, ketiga model menunjukkan performa akurasi yang cukup mirip di semua jenis feature. Pada feature BoW, model LSTM memberikan akurasi tertinggi (86.72%), sedikit unggul dibanding RNN (86.61%) dan SVM (85.81%). Untuk TF-IDF, performa ketiganya juga berdekatan, dengan LSTM tetap tertinggi (86.37%), diikuti SVM (86.30%) dan RNN (86.09%). Sementara itu, pada Word2Vec, justru SVM menghasilkan akurasi terbaik (85.56%), mengungguli RNN (85.11%) dan LSTM (85.04%).

In [ ]:
best_model_idx = df_akurasi['Akurasi'].idxmax()
best_model = df_akurasi.loc[best_model_idx]

In [ ]:
print("MODEL DENGAN AKURASI TERTINGGI")
print(f"Feature Extraction: {best_model['Feature']}")
print(f"Model Arsitektur   : {best_model['Model']}")
print(f"Akurasi            : {best_model['Akurasi (%)']}")
print(f"Nilai Akurasi      : {best_model['Akurasi']:.4f}")

MODEL DENGAN AKURASI TERTINGGI
Feature Extraction: BoW
Model Arsitektur   : LSTM
Akurasi            : 86.72%
Nilai Akurasi      : 0.8672


Model dengan performa terbaik diperoleh pada kombinasi feature extraction BoW dengan arsitektur LSTM, yang menghasilkan akurasi tertinggi sebesar 86.72% (0.8672).

# Menyimpan Model dan Feature Extract terbaik

In [ ]:
from google.colab import files

In [ ]:
model_lstm_bow.save('model_lstm_bow.h5')
files.download('model_lstm_bow.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import joblib
from google.colab import files

In [ ]:
joblib.dump(bow, 'bow_vectorizer.pkl')
files.download('bow_vectorizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>